# BabyGOT — Grounded Optimal-Transport Pretraining (Kaggle T4)

**BabyVLM Workshop (NeurIPS 2026)** — *"Toward Developmentally Plausible Multimodal Systems."*

This notebook trains a small vision-language model **from scratch on a single T4 GPU**
with a novel, mathematically-grounded objective:

- **Referential optimal transport (Sinkhorn)** aligns *patch* tokens with *word* tokens
  through a soft, doubly-stochastic coupling $P^\star$ — a generalisation of the global
  CLIP/CVCL (InfoNCE) and FILIP alignment losses.
- **Grounded summary tokens** preserve *where / how many*, fixing the "global-embedding
  bottleneck" of the BabyVLM / BabyVLM-V2 / LLaVA connector.
- A **token-wise gate** learns the content-word / function-word split with no supervision.

Full mathematics: `paper/paper.md`.  Read the recommended papers first:
BabyVLM (2504.09426), BabyVLM-V2 (2512.10932), Looking to Learn (2025.babylm-main.15),
LLaVA (2304.08485), BabyView (2406.10447), SAYCam (Open Mind 2022).

### 1. Environment (T4 = 16 GB VRAM)

The code needs only `torch` + `numpy`; Kaggle ships both.  Set the accelerator to
**GPU T4 x2** (or **CPU** — the `--tiny` smoke test runs either way).  Two ways to
get the repo:

- **Internet ON**: the cell below `git clone`s it for you.
- **Internet OFF**: upload this repo as a Kaggle *Dataset* named `babyvlm` and it
  will be found under `/kaggle/input/babyvlm` (the cell copies it into the writable
  working dir automatically).

Then the package is installed editable, so the run cells need **no `PYTHONPATH`**
and work regardless of the current directory.


In [ ]:
import os, shutil, subprocess, sys

SRC = 'src/babygot'
def have(p):
    return os.path.isdir(os.path.join(p, SRC))

# 1) Where is the repo?  (current dir, uploaded Dataset, or clone it)
if have('.'):
    REPO = '.'
elif have('/kaggle/input/babyvlm'):
    REPO = '/kaggle/input/babyvlm'        # read-only Kaggle Dataset
elif have('Baby-VLM'):
    REPO = 'Baby-VLM'
else:
    subprocess.run(['git', 'clone', 'https://github.com/Kylian07/Baby-VLM.git'],
                   check=True)
    REPO = 'Baby-VLM'

# 2) Kaggle Datasets are read-only -> copy into the writable working dir.
if REPO.startswith('/kaggle/input'):
    dst = 'Baby-VLM'
    if not have(dst):
        shutil.copytree(REPO, dst)
    REPO = dst

os.chdir(REPO)

# 3) Install the package (editable, offline-safe: no deps, no build isolation).
ok = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', '.',
     '--no-deps', '--no-build-isolation', '-q']).returncode == 0
if not ok:
    # Fallback: put src/ on the path for both Python and later `!` cells.
    os.environ['PYTHONPATH'] = os.getcwd() + '/src'
    sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
    print('(editable install failed — using PYTHONPATH=src fallback)')

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


### 2. Smoke test (tiny model, ~30 s)

Confirms the pipeline end-to-end before committing GPU time.

In [ ]:
!python -m babygot.run --method babygot --tiny


### 3. Full run on the T4 (base config, ~6M params)

Pretraining (from scratch) + instruction tuning + the 10-probe DevCV-lite suite.
The model trains in minutes on a T4.

In [ ]:
!python -m babygot.run --method babygot \
    --steps 4000 --n-train 4000 --n-eval 200 --save-dir runs


### 4. The paper's ablation table (all methods, same budget & seed)

- `babygot` — ours
- `global_clip` — CLIP/CVCL-style global InfoNCE + generative LM
- `babyllava` — BabyVLM-style single global token + autoregression
- `no_gate`, `no_ot` — component ablations

In [ ]:
!python -m babygot.run --all --small


### 5. Results + interpretability

Every run writes `runs/<method>.json` with per-probe accuracy plus two
interpretability metrics produced by the model itself:

- `analysis.mean_localization_error_patch` — distance between the OT coupling's
  centre of mass (the model's "pointing") and the true object, in patch units;
- `analysis.gate_content` / `analysis.gate_function` — the learned
  content/function-word gate.

In [ ]:
import json, glob
import matplotlib.pyplot as plt

rows = {}
for f in sorted(glob.glob('runs/*.json')):
    rows[f.split('/')[-1][:-5]] = json.load(open(f))

probes = [k for k in next(iter(rows.values())) if k != 'overall_choice_acc' and k != 'analysis']
fig, ax = plt.subplots(figsize=(12, 5))
for name, r in rows.items():
    vals = [r[p].get('acc', r[p].get('f1', 0.0)) for p in probes]
    ax.plot(probes, vals, marker='o', label=name)
ax.axhline(0.25, ls='--', c='grey', lw=1)
ax.set_ylim(0, 1); ax.legend(); ax.set_ylabel('accuracy / F1')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

for name, r in rows.items():
    a = r.get('analysis', {})
    if a:
        print(f"{name:12s}  loc_err={a.get('mean_localization_error_patch', float('nan')):.2f}"
              f"  gate_content={a.get('gate_content', 0):.3f}  gate_function={a.get('gate_function', 0):.3f}")

### 6. What to expect

- **Picture vocabulary** saturates quickly for *all* methods (naming is easy).
- **Localization, counting, spatial details** are near chance at the small
  demonstration budget; the *structural* point is that a mean-pooled connector
  is permutation-invariant and so has **zero capacity** for left/right/top/bottom,
  whereas the spatially-ordered summary gives the decoder non-zero capacity.
- The **OT coupling** exposes a word→region pointing map and the **gate** a
  content/function signal; both are first-class model outputs, but at the small
  budget they are not yet converged (see `paper/paper.md` §4.2 for the honest
  numbers).

For a deep dive, read `paper/paper.md` and `src/babygot/transport.py`.
